In [1]:

#helper class for  layer
import math
import numpy as np

class neuron_layer():

  def __init__(self,number_of_neurons,number_of_inputs_per_neuron,layer_name="hidden layer",activation="sigmoid"):
    if not isinstance(number_of_neurons,int) or number_of_neurons < 1: raise Exception("Number of neurons must be an integer "+self.layer_name)
    if not isinstance(number_of_inputs_per_neuron,int) or number_of_inputs_per_neuron<1 : raise Exception("Number of inputs per neuron must be an integer "+self.layer_name)

    self.layer_name = layer_name

    self.weights = np.random.randn(number_of_neurons, number_of_inputs_per_neuron) * 0.1
    self.biases = np.random.randn(number_of_neurons) * 0.1
    self.number_of_neurons = number_of_neurons
    self.number_of_inputs_per_neuron = number_of_inputs_per_neuron

    if activation == "sigmoid":
      #doesnt learn properly if all weiht bias same or uniformly distributed
      self.activation = lambda x:1/(1+np.exp(-x))
      self.activation_derivative = lambda y:y*(1-y)
    elif activation == "tanh":
      #tanh can cause vanshing gradient if all w=0 or all b =0
      self.activation = lambda x:np.tanh(x)
      self.activation_derivative = lambda y:1 - (y*y)
    else:
      raise Exception("Invalid Activation Function")

  def update_weights(self,delta_weights):
    delta_weights = np.array(delta_weights)
    if self.weights.shape != delta_weights.shape: raise Exception("Shape Mismatch In Neuron " +self.layer_name)
    self.weights = self.weights + delta_weights

  def update_biases(self, delta_biases):
    delta_biases = np.array(delta_biases)
    if self.biases.shape != delta_biases.shape: raise Exception("Shape Mismatch In Neuron " +self.layer_name,self.biases.shape,delta_biases.shape)
    self.biases = self.biases + delta_biases

  def predict_output(self,input:list):
    input = np.array(input)
    if input.shape != (self.number_of_inputs_per_neuron,): raise Exception("Shape Mismatch In Neuron " +self.layer_name)
    return np.array([self.activation(np.dot(w,input)+b) for w,b in zip(self.weights,self.biases)])

  def predict(self,input:list)->dict:

    input = np.array(input)
    if input.shape != (self.number_of_inputs_per_neuron,): raise Exception("Shape Mismatch In Neuron " +self.layer_name)

    outputs = np.array([self.activation(np.dot(w,input)+b) for w,b in zip(self.weights,self.biases)])
    derivatives = self.activation_derivative(outputs)
    return {
        "outputs": outputs,
        "derivatives": derivatives,
        "weights":self.weights.copy()
    }

In [8]:
activation = "sigmoid"
batch_execution = True

if activation == "sigmoid":
  data = [
      [ [0,1],1],
      [ [1,0],1],
      [ [0,0],0],
      [ [1,1],0],
  ]
elif activation == "tanh":
  data = [
      [ [-1,-1],-1],
      [ [1,-1],1],
      [ [-1,1],1],
      [ [1,1],-1],
  ]

alpha=0.3


output_layer = neuron_layer(1,2,"output layer",activation)
hidden_layer2 = neuron_layer(2,2,"hidden layer2",activation)


for epoch in range(50000):
  if batch_execution:
    d_output_layer = np.zeros(output_layer.weights.shape)
    d_bias_output_layer = np.zeros(output_layer.biases.shape)
    d_hidden_layer2 = np.zeros(hidden_layer2.weights.shape)
    d_bias_hidden_layer2 = np.zeros(hidden_layer2.biases.shape)

  for x,target in data:
    hidden_layer2_output = hidden_layer2.predict(x)
    output_layer_output = output_layer.predict(hidden_layer2_output["outputs"])

    error = target-output_layer_output["outputs"]

    output_layer_delta = error*output_layer_output["derivatives"]
    hidden_layer2_delta = np.sum([d*w for d,w in zip(output_layer_delta,output_layer.weights)] ,axis=0)*hidden_layer2_output["derivatives"]

    if not batch_execution:
      output_layer.update_weights([alpha*d*hidden_layer2_output["outputs"] for d in output_layer_delta])
      output_layer.update_biases(alpha*output_layer_delta)
      hidden_layer2.update_weights([alpha*d*np.array(x) for d in hidden_layer2_delta])
      hidden_layer2.update_biases(alpha*hidden_layer2_delta)

    if batch_execution:
      d_output_layer +=  np.array([alpha*d*hidden_layer2_output["outputs"] for d in output_layer_delta])
      d_bias_output_layer += alpha*output_layer_delta
      d_hidden_layer2 += np.array([alpha*d*np.array(x) for d in hidden_layer2_delta])
      d_bias_hidden_layer2 += alpha*hidden_layer2_delta

  if batch_execution:
    output_layer.update_weights(d_output_layer)
    output_layer.update_biases(d_bias_output_layer)
    hidden_layer2.update_weights(d_hidden_layer2)
    hidden_layer2.update_biases(d_bias_hidden_layer2)

for [each,t] in data:
  print("final",each,output_layer.predict_output(hidden_layer2.predict_output(each)))

final [0, 1] [0.99084394]
final [1, 0] [0.99081769]
final [0, 0] [0.00879827]
final [1, 1] [0.01125735]
